In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

## Local MCP server

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [3]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='43a9e84e-8873-45a9-bef6-23c60d4a59de'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eda49-d46f-7021-9e68-f070ccba29aa-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '23da5251-39c1-486e-823a-fc1e36096496', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 194, 'output_tokens': 22, 'total_tokens': 216, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langchain-mcp-adapters library",\n  "follow_up_questions": null,\n  "answer": null,\n  "images":

## Online MCP

In [11]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [15]:
agent = create_agent(
    model=model,
    # tools=tools,
)

In [16]:
question = HumanMessage(content="What time is it in india?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it in india?', additional_kwargs={}, response_metadata={}, id='d9b6f90c-0e6a-459d-81fd-4c52859d26d7'),
              AIMessage(content='I cannot provide the current time in India or any other location. My knowledge is based on the data I was trained on, and I don\'t have access to real-time information like the current time.\n\nTo find out the current time in India, you can:\n\n*   **Search online:** A quick search on Google or any other search engine for "time in India" will give you the current time.\n*   **Check your device:** Most smartphones and computers have a clock that can display the time in different time zones.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eda54-3173-76a3-a5f0-a78f27d635ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 110, 'total_t

In [21]:
print(response['messages'][-1].content)

I cannot provide the current time in India or any other location. My knowledge is based on the data I was trained on, and I don't have access to real-time information like the current time.

To find out the current time in India, you can:

*   **Search online:** A quick search on Google or any other search engine for "time in India" will give you the current time.
*   **Check your device:** Most smartphones and computers have a clock that can display the time in different time zones.
